# Strategy Template — Multi-Year Simple Momentum

Walk-forward research via `ccquant.strategy`. Default: **`cs_mom_simple`**
(pure CS momentum on the daily OHLCV panel — no OI / macro regime).

**Pipeline:** PIT features → weekly L/S → costs → purged walk-forward → scale gates.

**Prereqs (multi-year)**
```bash
uv run ccquant sync universe --size 50
uv run ccquant sync backfill --interval 1d --full --force --top 50
uv run dbt run --select fct_ohlcv_daily --full-refresh --project-dir dbt --profiles-dir dbt
uv run ccquant research run --strategy cs_mom_simple
```

Contract: [`documentation/strategy_research.md`](../documentation/strategy_research.md).

In [ ]:
from __future__ import annotations

import os
import warnings
from pathlib import Path

import plotly.express as px
import plotly.graph_objects as go
import polars as pl
from dotenv import load_dotenv
from IPython.display import display

from ccquant.strategy import load_strategy_panel, panel_history, run_strategy_detailed
from ccquant.strategy.spec import default_strategy_config_path, load_strategy_config

warnings.filterwarnings("ignore", category=FutureWarning)

_root = Path.cwd()
while not (_root / "pyproject.toml").exists() and _root.parent != _root:
    _root = _root.parent
os.chdir(_root)
load_dotenv(_root / ".env", override=True)

DB = Path(os.environ.get("CCQUANT_DB", _root / "data" / "ccquant.duckdb")).resolve()
STRATEGY = "cs_mom_simple"
CFG_PATH = default_strategy_config_path(STRATEGY)
print(f"DB: {DB}  exists={DB.exists()}")
print(f"Config: {CFG_PATH}")

## 1. Load panel + run strategy

Uses `panel: daily` from the YAML. Falls back to a long synthetic panel when the DB is missing/short so the notebook still runs top-to-bottom.

In [ ]:
import math
from datetime import date, timedelta

cfg = load_strategy_config(CFG_PATH)
panel: pl.DataFrame | None = None
source = "synthetic"

if DB.exists():
    try:
        panel = load_strategy_panel(DB, cfg)
        hist = panel_history(panel)
        if panel.height > 0:
            source = cfg.panel
            print(
                f"Loaded {panel.height:,} rows · {hist.n_symbols} symbols · "
                f"{hist.date_min} → {hist.date_max} ({hist.n_calendar_days} days)"
            )
            if not hist.is_multiyear:
                print(
                    "WARNING: history < 365 days — not multi-year. "
                    "Run sync backfill --interval 1d --full --force --top 50"
                )
    except Exception as exc:  # noqa: BLE001 — notebook degrade path
        print(f"panel load failed: {exc}")
        panel = None

if panel is None or panel.height == 0:

    def _synth(n_days: int = 900, symbols: tuple[str, ...] = ("AAA", "BBB", "CCC", "DDD", "EEE", "BTC")) -> pl.DataFrame:
        rows: list[dict[str, object]] = []
        start = date(2020, 1, 1)
        dates = [start + timedelta(days=i) for i in range(n_days)]
        for si, sym in enumerate(symbols):
            price = 100.0 * (1.0 + 0.1 * si)
            mu = 0.001 * (si + 1)
            for di, d in enumerate(dates):
                price *= 1.0 + mu + 0.01 * math.sin(di / 7.0 + si)
                rows.append(
                    {
                        "symbol": sym,
                        "date": d,
                        "open": price,
                        "high": price * 1.01,
                        "low": price * 0.99,
                        "close": price,
                        "volume": 1_000_000.0 / price,
                    }
                )
        return pl.DataFrame(rows).with_columns(pl.col("date").cast(pl.Date))

    panel = _synth()
    print("Using synthetic multi-year panel (no DB panel available)")

run = run_strategy_detailed(panel=panel, config=cfg, write_dir=_root / "data" / "research")
report = run.report
daily = run.daily
print(
    f"strategy={report.strategy_name} hash={report.config_hash} passed={report.passed} "
    f"folds={report.n_folds}"
)
print(
    f"history={report.data_min_date} → {report.data_max_date} "
    f"({report.n_calendar_days} days) capacity_usd={report.capacity_usd:,.0f}"
)
if report.gate_reasons:
    for r in report.gate_reasons:
        print(f"  gate: {r}")
display(pl.DataFrame({"metric": list(report.oos_metrics.keys()), "value": list(report.oos_metrics.values())}))

## 2. Equity curve & drawdown

In [ ]:
eq = daily.select(["date", "equity_net", "equity_gross", "net_ret"]).drop_nulls()
fig = go.Figure()
fig.add_trace(go.Scatter(x=eq["date"], y=eq["equity_net"], name="equity_net", line=dict(width=2)))
fig.add_trace(go.Scatter(x=eq["date"], y=eq["equity_gross"], name="equity_gross", line=dict(width=1, dash="dot")))
fig.update_layout(title=f"{report.strategy_name} — equity (net vs gross)", template="plotly_white", height=420)
fig.show()

dd = eq.with_columns(
    (pl.col("equity_net") / pl.col("equity_net").cum_max() - 1.0).alias("drawdown")
)
fig_dd = px.area(dd.to_pandas(), x="date", y="drawdown", title="Net drawdown")
fig_dd.update_layout(template="plotly_white", height=320)
fig_dd.show()

## 3. Fold Sharpes (multi-year OOS windows)

In [ ]:
print("OOS fold count:", report.n_folds)
if report.fold_metrics:
    fold_df = pl.DataFrame(list(report.fold_metrics))
    cols = [c for c in ("fold", "net_sharpe", "gross_sharpe", "ir_ew", "max_drawdown", "avg_turnover", "n_days") if c in fold_df.columns]
    display(fold_df.select(cols))
    if "net_sharpe" in fold_df.columns:
        fig_f = px.bar(fold_df.to_pandas(), x="fold", y="net_sharpe", title="OOS net Sharpe by fold")
        fig_f.update_layout(template="plotly_white", height=320)
        fig_f.show()
else:
    print("No purged folds (history shorter than train+embargo+test).")

## 4. Scale-gate verdict

Pass requires OOS net Sharpe > 0, IR(EW) > 0, and capacity ≥ target notional. A FAIL on a correct multi-year sample means no edge — not a harness bug.

In [ ]:
verdict = "PASSED" if report.passed else "FAILED"
print(f"Scale gates: {verdict}")
print(f"net_sharpe={report.oos_metrics.get('net_sharpe')}")
print(f"ir_ew={report.oos_metrics.get('ir_ew')}")
print(f"capacity_usd={report.capacity_usd:,.0f} vs target={report.target_notional_usd:,.0f}")
print("Report JSON under data/research/ (gitignored).")